# 03: Double Pulse Test (DPT) transients

Clamped-inductive two-pulse test. The vendor golden model and the Tier-2 ROM are driven with the **same** per-device conditions from `config/inputs/<vendor>.yaml` → `dpt`. Timing/energy figures come from `metrics.switching` / `metrics.loss`; the simulator wall time is measured directly (three runs per model, median reported). No synthetic waveforms or invented speedups.

In [ ]:
import sys, os, json
from pathlib import Path
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT)); os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
import yaml
from common.plotting import setup_style, plot_dual_axis, plot_waveforms
from common.types import WaveformData
from simulation.testbenches.double_pulse import run_double_pulse, extract_dpt_metrics
from simulation.testbenches.capacitance import CapacitanceTestbench
from approximation.charge_model import integrate_charge
from parser.netlist import resolve_pin_connections
setup_style()

VENDORS = ('irfs7730pbf', 'csd16412q5a', 'epc2216', 'cpm2-1200')
FIG = REPO_ROOT / 'figures'; CMP = REPO_ROOT / 'reports' / 'comparison'
N_REPEATS = 3

In [ ]:
def clean(v):
    return None if (v is None or (isinstance(v, float) and not np.isfinite(v))) else float(v)

summary = []
for vendor in VENDORS:
    cfg = yaml.safe_load((REPO_ROOT / 'config' / 'inputs' / f'{vendor}.yaml').read_text(encoding='utf-8'))
    dpt = cfg['dpt']; cond = cfg['conditions']
    roles = resolve_pin_connections(REPO_ROOT / cfg['library'], cfg['subcircuit'],
                                    cfg['pins']['pin_order_semantic'])
    keys = ('vbus', 'vgs_on', 'i_load_target', 'l_load', 't1_on', 't1_width',
            't2_on', 't2_width', 'edge_time', 't_end', 'max_step')
    args = {k: dpt[k] for k in keys}

    # Eoss only when the datasheet cap sweep actually covers the DPT bus.
    _, coss, _ = CapacitanceTestbench(f_meas=cond.get('F_meas')).run(
        REPO_ROOT / cfg['library'], cfg['subcircuit'],
        vds_min=cond['Vds_cap_min'], vds_max=cond['Vds_cap_max'],
        points=cond['Cap_points'], pin_roles=roles)
    eoss = float('nan')
    if dpt['vbus'] <= max(coss.x_data):
        cm = integrate_charge(coss.x_data, coss.y_data, v_ref=dpt['vbus'])
        eoss = cm.e[-1] * 1e6

    def run(lib, sub, pins):
        return run_double_pulse(lib, sub, pins, temperature=cond.get('temperature', 25.0), **args)

    golden, gfail = [], ''
    try:
        for _ in range(N_REPEATS):
            golden.append(run(REPO_ROOT / cfg['library'], cfg['subcircuit'], roles))
    except Exception as exc:
        gfail = f'{type(exc).__name__}: {exc}'[:160]

    rom_lib = REPO_ROOT / 'reports' / 'models' / f'{vendor}_PIECESPICE_T2.lib'
    rom = [run(rom_lib, rom_lib.stem, ['drain', 'gate', 'source'])
           for _ in range(N_REPEATS)]

    gm = extract_dpt_metrics(golden[0], eoss) if golden else None
    tm = extract_dpt_metrics(rom[0], eoss)
    gt = float(np.median([r.wall_clock_s for r in golden])) if golden else None
    tt = float(np.median([r.wall_clock_s for r in rom]))
    speedup = gt / tt if gt else None

    figdir = FIG / vendor; figdir.mkdir(parents=True, exist_ok=True)
    plot_dual_axis(rom[0].time * 1e6, rom[0].v_ds, rom[0].i_d,
                   left_label='Vds [V]', right_label='Id [A]',
                   title=f'{vendor}: DPT waveforms (Tier-2 ROM)',
                   save_path=figdir / 'dpt_waveforms.png')
    gate_wfs = [WaveformData(name='Tier-2 Vgs', time=rom[0].time.tolist(),
                             values=rom[0].v_gs.tolist(), unit='V')]
    if golden:
        gate_wfs.insert(0, WaveformData(name='Golden Vgs',
                                        time=golden[0].time.tolist(),
                                        values=golden[0].v_gs.tolist(), unit='V'))
    plot_waveforms(gate_wfs, title=f'{vendor}: gate drive',
                   save_path=figdir / 'dpt_gate.png')

    outdir = CMP / vendor; outdir.mkdir(parents=True, exist_ok=True)
    payload = {'schema': 'piecespice.dpt/1.0', 'vendor': vendor,
               'conditions': args, 'repeats': N_REPEATS,
               'golden_failure': gfail,
               'golden_metrics': {k: clean(v) for k, v in gm.items()} if gm else None,
               'tier2_metrics': {k: clean(v) for k, v in tm.items()},
               'wall_clock_s': {'golden_median': clean(gt),
                                'tier2_median': clean(tt),
                                'speedup': clean(speedup)}}
    (outdir / 'dpt_metrics.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')

    row = {'vendor': vendor, 'golden_failure': gfail[:60] if gfail else '',
           'golden_s': clean(gt), 'rom_s': clean(tt), 'speedup': clean(speedup),
           'eoss_uj': clean(eoss)}
    for k in ('td_on_ns', 'tr_ns', 'td_off_ns', 'tf_ns',
              'dv_dt_max_v_per_ns', 'di_dt_max_a_per_ns', 'e_on_uj', 'e_off_uj'):
        row[f'gold_{k}'] = clean(gm[k]) if gm else None
        row[f'rom_{k}'] = clean(tm[k])
    summary.append(row)
    print(vendor, 'golden s=', clean(gt), 'rom s=', clean(tt),
          'speedup=', clean(speedup), 'failure=', gfail[:60], flush=True)

## Results

Wall time is the measured `SpiceEngine.run_transient` duration (LTspice subprocess included), median of three runs per model. `NaN` is reported when a level crossing is absent — never zero. CPM2 golden DPT failure is the documented vendor turn-off abort (VM-10), so no speedup is claimed for that device.

In [ ]:
df = pd.DataFrame(summary)
display(df[['vendor', 'golden_s', 'rom_s', 'speedup', 'eoss_uj',
             'gold_td_on_ns', 'rom_td_on_ns', 'gold_e_on_uj', 'rom_e_on_uj',
             'golden_failure']])
print('Saved per-device DPT tables under reports/comparison/<vendor>/dpt_metrics.json')